# Complete (gap-free) AIRS CO dataset

Same loading and interpolation steps as `../detrend.ipynb`: monthly cropped AIRS files are
averaged over orbit passes, converted with the CO/CO2 molar-mass ratio, stacked onto a full
Jan 2003 – Dec 2025 monthly calendar, and missing months are filled by per-pixel linear
interpolation in time. The result (before any detrending) is saved as `complete_data.nc`.

In [1]:
# Setup: config + imports
import glob
import os
import numpy as np
import xarray as xr
import pandas as pd

# Config
CROP_OUT_DIR = "../../Data/AIRS-cropped"
VARNAME      = "co_mmr_midtrop"
MW_RATIO     = 28.01 / 44.01          # CO / CO2 molar-mass conversion
OUT_NC       = "complete_data.nc"

# Balanced window: 23 whole years
START_TS = pd.Timestamp("2003-01-01")
END_TS   = pd.Timestamp("2025-12-01")

## Interpolation to fill gaps in the data (2 months)

In [2]:
# Build gap-free monthly maps over the window, then the spatial-mean series.
crop_files = sorted(glob.glob(os.path.join(CROP_OUT_DIR, "*.nc")))
print(f"Found {len(crop_files)} cropped files")

# Load each month's (lat, lon) map: mean over orbit passes, ratio-corrected.
obs_maps = {}
lons = lats = None
for f in crop_files:
    stem = os.path.splitext(os.path.basename(f))[0]
    date = pd.Timestamp(f"{stem[:4]}-{stem[4:]}-01")
    if not (START_TS <= date <= END_TS):
        continue
    ds_g = xr.open_dataset(f, decode_timedelta=False)
    if lons is None:
        lons = ds_g["lon"].values
        lats = ds_g["lat"].values
    obs_maps[date] = ds_g[VARNAME].mean(dim="orbit_pass", skipna=True).values * MW_RATIO
    ds_g.close()

n_lat, n_lon = len(lats), len(lons)

# Full monthly calendar over the window.
months = pd.date_range(START_TS, END_TS, freq="MS")

# Stack into a (time, lat, lon) cube; missing months stay all-NaN.
raw_maps = np.full((len(months), n_lat, n_lon), np.nan)
has_obs = np.zeros(len(months), dtype=bool)
for i, d in enumerate(months):
    if d in obs_maps:
        raw_maps[i] = obs_maps[d]
        has_obs[i] = True
is_interp = ~has_obs

# Interpolate maps in time, per pixel, to fill missing months.
final_maps = xr.DataArray(
    raw_maps, dims=("time", "lat", "lon"),
    coords={"time": months, "lat": lats, "lon": lons},
)
final_maps = (final_maps.interpolate_na(dim="time", method="linear")
                        .bfill("time").ffill("time"))

# Time series = spatial mean of each map.
ts_dates = months
ts_co    = final_maps.mean(dim=("lat", "lon"), skipna=True).values

# Diagnostics.
n_pix = n_lat * n_lon
filled_dates = [d.strftime('%Y-%m') for d in months[is_interp]]
print(f"Calendar months   : {len(months)}   ({START_TS:%b %Y} - {END_TS:%b %Y})")
print(f"Observed maps      : {has_obs.sum()}")
print(f"Reconstructed maps : {is_interp.sum()}  -> {filled_dates}")
print(f"Grid per map       : {n_lat} lat x {n_lon} lon = {n_pix} pixels")
for i in np.where(is_interp)[0]:
    n_valid_orig = int(np.isfinite(raw_maps[i]).sum())
    n_valid_new  = int(np.isfinite(final_maps.values[i]).sum())
    print(f"   {months[i].strftime('%Y-%m')}: filled {n_valid_new - n_valid_orig} "
          f"pixels  ->  map mean = {ts_co[i]:.4e} kg/kg")

n_dead = int(np.isnan(final_maps.values).all(axis=0).sum())
print(f"Pixels never observed (left NaN): {n_dead}")
print(f"Date range : {ts_dates[0].date()}  ->  {ts_dates[-1].date()}")
print(f"CO range   : {np.nanmin(ts_co):.4e} - {np.nanmax(ts_co):.4e}  kg/kg")

Found 279 cropped files


Calendar months   : 276   (Jan 2003 - Dec 2025)
Observed maps      : 274
Reconstructed maps : 2  -> ['2023-09', '2025-12']
Grid per map       : 36 lat x 61 lon = 2196 pixels
   2023-09: filled 2196 pixels  ->  map mean = 8.5295e-08 kg/kg
   2025-12: filled 2196 pixels  ->  map mean = 7.6991e-08 kg/kg
Pixels never observed (left NaN): 0
Date range : 2003-01-01  ->  2025-12-01
CO range   : 7.6991e-08 - 1.1788e-07  kg/kg


## Save the complete record to `complete_data.nc`

In [3]:
complete_data = final_maps.transpose("time", "lat", "lon").rename(VARNAME)
complete_data.attrs = {
    "long_name": "AIRS CO mid-troposphere mass mixing ratio to dry air (gap-filled)",
    "units": "kg kg-1",
    "note": (f"Mean over orbit passes x {MW_RATIO:.6f} (CO/CO2 molar-mass ratio); "
             f"missing months filled by per-pixel linear interpolation in time: "
             f"{', '.join(filled_dates)}"),
}
# keep track of which months were filled, as a time coordinate
complete_data = complete_data.assign_coords(interpolated=("time", is_interp.astype("int8")))
complete_data["interpolated"].attrs = {"long_name": "1 = month filled by interpolation, 0 = observed"}

complete_data.to_netcdf(OUT_NC)
print("Saved ->", os.path.abspath(OUT_NC))

Saved -> D:\AIRS\harmonic-analysis\Scripts\climatology\complete_data.nc


## Check the saved file

In [4]:
chk = xr.open_dataarray(OUT_NC)
print(chk)
print("\nNaNs           :", int(np.isnan(chk.values).sum()))
print("Identical to final_maps:", np.array_equal(chk.values, final_maps.values))

# Sanity check: detrending this file the same way as detrend.ipynb reproduces ../detr_field.nc
yr   = np.arange(chk.sizes["time"]) / 12.0
Y    = chk.values
yr_c = yr - yr.mean()
slope = np.tensordot(yr_c, Y - Y.mean(axis=0, keepdims=True), axes=(0, 0)) / np.sum(yr_c**2)
inter = Y.mean(axis=0) - slope * yr.mean()
detr  = Y - (inter[None] + slope[None] * yr[:, None, None])

detr_ref = xr.open_dataarray("../detr_field.nc")
print(f"Max |detrended(complete_data) - detr_field.nc| = {np.abs(detr - detr_ref.values).max():.3e} kg/kg")
chk.close(); detr_ref.close()

<xarray.DataArray 'co_mmr_midtrop' (time: 276, lat: 36, lon: 61)> Size: 5MB
[606096 values with dtype=float64]
Coordinates:
  * time          (time) datetime64[ns] 2kB 2003-01-01 2003-02-01 ... 2025-12-01
  * lat           (lat) float32 144B 14.5 15.5 16.5 17.5 ... 46.5 47.5 48.5 49.5
  * lon           (lon) float32 244B -126.5 -125.5 -124.5 ... -68.5 -67.5 -66.5
    interpolated  (time) int8 276B ...
Attributes:
    long_name:  AIRS CO mid-troposphere mass mixing ratio to dry air (gap-fil...
    units:      kg kg-1
    note:       Mean over orbit passes x 0.636446 (CO/CO2 molar-mass ratio); ...

NaNs           : 0
Identical to final_maps: True
Max |detrended(complete_data) - detr_field.nc| = 0.000e+00 kg/kg
